In [1]:
import cadquery as cq
from cadquery import exporters
import math
from jupyter_cadquery import *

from jupyter_cadquery.replay import replay, enable_replay, disable_replay

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

versions()

Overwriting auto display for cadquery Workplane and Shape

Versions:
- jupyter_cadquery  4.0.2
- cad_viewer_widget 3.0.2
- open cascade      7.7.2.1



In [2]:
cv = open_viewer("Stud", cad_width=800, height=600)
set_default_viewer("Stud")

In [3]:
# Stud
stud_width = 3.0
stud_depth = 4.0
stud_length = 112.0       # shoulder-to-shoulder length

# Stub tenons
tenon_width = 3.0        # along X
tenon_thickness = 1.5    # along Y
tenon_length = 2.0       # projection into sill/girder

# Horizontal members
sill_width = 48.0        # along X
sill_depth = 8.0         # along Y
sill_height = 10.0        # along Z

girder_width = 48.0
girder_depth = 4.0
girder_height = 6.0

# Mortise fit
clearance = 0.0625

In [4]:
stud_body = (
    cq.Workplane("XY")
    .box(
        stud_width,
        stud_depth,
        stud_length,
        centered=(True, True, False),
    )
)

In [5]:
bottom_tenon = (
    cq.Workplane("XY")
    .workplane(offset=-tenon_length)
    .box(
        tenon_width,
        tenon_thickness,
        tenon_length,
        centered=(True, True, False),
    )
)

stud = stud_body.union(bottom_tenon)

In [6]:
top_tenon = (
    cq.Workplane("XY")
    .workplane(offset=stud_length)
    .box(
        tenon_width,
        tenon_thickness,
        tenon_length,
        centered=(True, True, False),
    )
)

stud = stud.union(top_tenon)

In [7]:
sill_y = (sill_depth - stud_depth) / 2

sill = (
    cq.Workplane("XY")
    .workplane(offset=-sill_height)
    .box(
        sill_width,
        sill_depth,
        sill_height,
        centered=(True, True, False),
    ).translate((0, sill_y, 0))
)





In [8]:
bottom_mortise = (
    cq.Workplane("XY")
    .workplane(offset=-tenon_length)
    .box(
        tenon_width + clearance,
        tenon_thickness + clearance,
        tenon_length,
        centered=(True, True, False),
    )
)

sill_cut = sill.cut(bottom_mortise)

In [9]:
girder = (
    cq.Workplane("XY")
    .workplane(offset=stud_length)
    .box(
        girder_width,
        girder_depth,
        girder_height,
        centered=(True, True, False),
    )
)

In [10]:
top_mortise = (
    cq.Workplane("XY")
    .workplane(offset=stud_length)
    .box(
        tenon_width + clearance,
        tenon_thickness + clearance,
        tenon_length,
        centered=(True, True, False),
    )
)

girder_cut = girder.cut(top_mortise)

In [11]:
show_object(sill_cut, name="sill")
show_object(stud, name="stud")
show_object(girder_cut, name="girder")

+
++
+++


In [12]:
print("Stud:", stud.val().BoundingBox())
print("Sill:", sill_cut.val().BoundingBox())
print("Girder:", girder_cut.val().BoundingBox())

Stud: <cadquery.occ_impl.geom.BoundBox object at 0x7fb349f73b00>
Sill: <cadquery.occ_impl.geom.BoundBox object at 0x7fb349f73b00>
Girder: <cadquery.occ_impl.geom.BoundBox object at 0x7fb349f73b00>
